# Lab 2 essentials: the same pieces in three vector databases

**Time: about 45 minutes. Click Run all, then read each step while it runs. Do Lab 1 essentials first.**

**What you will do.** In Lab 1 you cut FreshCart's seven documents into pieces, labelled them and
searched them in a simple store that lives in the notebook's memory. Real systems use a **vector
database**: a database built to store embeddings and find the closest ones fast. Here you put the same
pieces into three popular ones, **Chroma**, **Qdrant** and **FAISS**, with almost the same code, check
that they find the right pieces, and find out where they really differ. Along the way you will find
two problems that no error message tells you about.

**The libraries, and what each one does.** Nothing else is used: no lab-specific code.

| Library | What it does here | Step | Docs |
|---|---|---|---|
| `langchain-community`, `langchain-text-splitters`, `pypdf`, `docx2txt`, `python-pptx`, `openpyxl` | load and cut the documents, exactly as in Lab 1 | 1 | see Lab 1 essentials |
| `fastembed` | the same free embedding model as Lab 1, BGE small | 1 | [qdrant.github.io/fastembed](https://qdrant.github.io/fastembed/) |
| `numpy` | exact search: compares the question with every piece, the answer key | 2 | [numpy.org](https://numpy.org/doc/stable/) |
| `langchain-chroma` + `chromadb` | LangChain's connector to **Chroma**, a database that runs inside Python | 3 | [docs.trychroma.com](https://docs.trychroma.com/) |
| `langchain-qdrant` + `qdrant-client` | LangChain's connector to **Qdrant**, here running in memory | 3 | [qdrant.tech/documentation](https://qdrant.tech/documentation/) |
| `langchain-community` + `faiss-cpu` | LangChain's connector to **FAISS**, Meta's search library, which runs inside Python | 3 | [faiss.ai](https://faiss.ai/) |

To see every setting a library offers, run the **Look it up yourself** cell at the end, or type
`help(name)` in any cell.

**Four words used throughout.**

- A **vector database** stores pieces of text with their embeddings and finds the ones closest to a
  question. Chroma, Qdrant and FAISS are three of them (FAISS is strictly a search library, not a
  full database, but it does the same job here).
- **Exact search** compares the question with every single piece. Always right, slow when there are
  millions of pieces. It is the **answer key** here.
- **Recall@3** is the share of the answer key's top 3 that a database also returns in its top 3. 1.0
  means it found all of them.
- A **filter** limits a search to pieces whose labels match, for example only current documents.

## Step 0. Install the libraries

**What this does.** Installs the libraries from the table above, then restarts Python so the new
versions load. It takes one to two minutes. The restart is expected.

In [ ]:
# Lab 1's libraries: loaders, splitter, file readers and the embedding model.
# langchain-chroma, langchain-qdrant: LangChain's connectors to Chroma and Qdrant.
# faiss-cpu: FAISS itself; its LangChain connector is in langchain-community.
%pip install -q langchain-core langchain-community langchain-text-splitters pypdf docx2txt python-pptx openpyxl fastembed langchain-chroma langchain-qdrant faiss-cpu

In [ ]:
# Restart Python so the libraries installed above are the ones that get loaded.
dbutils.library.restartPython()

In [ ]:
# os lets us set settings for this session; warnings controls which notices are shown.
import os, warnings

# Hide notices that are not errors, such as messages about packages being renamed.
warnings.filterwarnings("ignore")

# Turn off the download progress bars of the embedding model.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
print("ready")

## Step 1. The pieces from Lab 1, and the embedding model

**What this does.** Repeats Lab 1 in one cell: copy the seven documents, read them, cut them into
pieces of 500 characters, and label each piece with its file, year and whether it is current. Each
piece keeps only those three labels, because the databases are strict about labels (Step 3 says why).
Then it loads the same embedding model as Lab 1.

**What you should see.** `21 pieces`, the three labels of one piece, and `384 numbers per embedding`.

In [ ]:
# The course files: the documents, the warehouse and the test questions, from this lab's GitHub
# repository. They are downloaded once and kept in one zip file, PACK_ZIP, that later steps read.
import io, pathlib, urllib.request, zipfile

PACK_ZIP = "/tmp/freshcart_course_files.zip"
if not pathlib.Path(PACK_ZIP).exists():
    repo = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://github.com/mogupta84/ai-engineer-data-labs/archive/refs/heads/main.zip", timeout=120).read()))
    with zipfile.ZipFile(PACK_ZIP, "w") as out:
        for name in repo.namelist():
            inside = name.split("/", 1)[1] if "/" in name else ""     # drop the repository folder name
            if inside.startswith("data/") and not name.endswith("/"):
                out.writestr(inside, repo.read(name))
print("course files ready:", PACK_ZIP)

In [ ]:
# The same imports as Lab 1 essentials.
import zipfile, pathlib
import pandas as pd
from pptx import Presentation
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import FastEmbedEmbeddings

# Copy the seven documents out of the course zip file into a folder for this session.
DOCS = pathlib.Path("/tmp/lab02_essentials/docs")
DOCS.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACK_ZIP) as z:
    for name in z.namelist():
        if name.startswith("data/raw/") and not name.endswith("/"):
            (DOCS / pathlib.Path(name).name).write_bytes(z.read(name))

# Read all four formats into LangChain Documents, as in Lab 1 Steps 2 and 3.
docs = []
for f in sorted(DOCS.glob("*.pdf")):
    docs += PyPDFLoader(str(f)).load()
docs += Docx2txtLoader(str(DOCS / "support_sla_matrix.docx")).load()
for slide in Presentation(DOCS / "support_qbr_q1_2024.pptx").slides:
    text = "\n".join(shape.text_frame.text for shape in slide.shapes if shape.has_text_frame)
    docs.append(Document(page_content=text, metadata={"source": "support_qbr_q1_2024.pptx"}))
table = pd.read_excel(DOCS / "refund_claims_march_2024.xlsx")
docs.append(Document(page_content=table.to_string(index=False), metadata={"source": "refund_claims_march_2024.xlsx"}))

# Cut into pieces of at most 500 characters, as in Lab 1 Step 4, and drop empty pieces.
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
pieces = [p for p in splitter.split_documents(docs) if p.page_content.strip()]

# Replace each piece's labels with just our three. The loaders add others (page, producer, ...)
# that differ from file to file, and some databases refuse labels that are not the same on every piece.
for p in pieces:
    file = pathlib.Path(p.metadata["source"]).name
    p.metadata = {"file": file,
                  "year": 2022 if "2022" in file else 2024,
                  "current": file != "refund_policy_v2_2022.pdf"}

print(len(pieces), "pieces")
print("labels of one piece:", pieces[0].metadata)

# The same embedding model as Lab 1.
embedder = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")
print(len(embedder.embed_query("test")), "numbers per embedding")

## Step 2. Exact search: the answer key

**What this does.** Before trusting any database, build the answer key the slow, certain way: turn the
question and every piece into embeddings, and compare the question with every single piece. The three
pieces with the highest **cosine similarity** (1 means the same meaning, 0 means unrelated) are the
right answer.

**What you should see.** The same three as Lab 1, Step 6: the replaced 2022 policy first at 0.877,
then two 2024 pieces at 0.791 and 0.730. This list is what every database below is checked against.

In [ ]:
# numpy does arithmetic on whole lists of numbers at once.
import numpy as np

# Embed every piece once. embed_documents takes a list of texts and returns one embedding each.
piece_vectors = np.array(embedder.embed_documents([p.page_content for p in pieces]))

# exact_top3: the 3 pieces closest to a question, found by comparing it with every piece.
def exact_top3(question):
    # Embed the question.
    q = np.array(embedder.embed_query(question))
    # Cosine similarity with every piece: multiply and add up, then divide by both lengths.
    scores = piece_vectors @ q / (np.linalg.norm(piece_vectors, axis=1) * np.linalg.norm(q))
    # argsort sorts from low to high; [::-1] flips it; [:3] keeps the best three positions.
    best = np.argsort(scores)[::-1][:3]
    # Return each piece's position in the list and its score.
    return [(int(i), float(scores[i])) for i in best]

# The question from Lab 1, Step 6.
question = "How many working days until an approved refund is credited?"
for i, score in exact_top3(question):
    print(f"{score:.3f}  piece {i:>2}  {pieces[i].metadata['file']}")

## Step 3. Put the same pieces into three databases

**What this does.** LangChain gives every database the same method, `from_documents`: hand it the
pieces and the embedding model, and it embeds the pieces and stores them. Only the class name and its
connection setting change from one database to the next. The cell times each load.

- **Chroma** runs inside Python. `collection_metadata` tells it to compare by cosine.
- **Qdrant** runs in memory here, `location=":memory:"`. In production it is a separate server.
- **FAISS** runs inside Python with no server at all. By default it measures **distance**, not
  similarity, which Step 4 makes visible.

**What you should see.** Three lines, one per database, each with 21 pieces stored and how many
seconds it took, a few seconds each. Most of that time is the embedding model, not the database.

In [ ]:
# time.perf_counter is a stopwatch.
import time
from langchain_chroma import Chroma
from langchain_qdrant import QdrantVectorStore
from langchain_community.vectorstores import FAISS

stores = {}

# CHROMA. collection_metadata={"hnsw:space": "cosine"} makes it compare by cosine, like our answer key.
start = time.perf_counter()
stores["Chroma"] = Chroma.from_documents(pieces, embedder, collection_name="lab02",
                                         collection_metadata={"hnsw:space": "cosine"})
print(f"Chroma : stored {len(pieces)} pieces in {time.perf_counter() - start:.1f} s")

# QDRANT. location=":memory:" keeps everything in this notebook's memory.
start = time.perf_counter()
stores["Qdrant"] = QdrantVectorStore.from_documents(pieces, embedder, location=":memory:", collection_name="lab02")
print(f"Qdrant : stored {len(pieces)} pieces in {time.perf_counter() - start:.1f} s")

# FAISS. No connection setting at all: the index lives in this notebook's memory.
# We leave its default way of comparing, which is distance (lower means closer).
start = time.perf_counter()
stores["FAISS"] = FAISS.from_documents(pieces, embedder)
print(f"FAISS  : stored {len(pieces)} pieces in {time.perf_counter() - start:.1f} s")

## Step 4. Ask all three the same question

**What this does.** Sends the Step 2 question to each database with the same method,
`similarity_search_with_score`, and prints the three pieces each one returns, with its score.

**What you should see.** All three return the same three pieces in the same order as the answer key,
but with three different scores:

- **Qdrant:** 0.877, 0.791, 0.730. Cosine similarity, exactly the answer key's. Higher means closer.
- **Chroma:** 0.123, 0.209, 0.270. Cosine **distance**, which is 1 minus the similarity. Lower means closer.
- **FAISS:** 0.246, 0.417, 0.541. A different distance (squared straight-line distance, here twice
  Chroma's). Lower means closer.

Same ranking, three scales. A rule such as "refuse when the score is below 0.5" means three different
things on these three databases, and on Chroma and FAISS it would refuse the best answers.

In [ ]:
# The same call on every database: the question, and how many results to return.
for name, store in stores.items():
    print(name)
    for doc, score in store.similarity_search_with_score(question, k=3):
        print(f"   score {score:.3f}   {doc.metadata['file']}   year {doc.metadata['year']}")
    print()

# The answer key from Step 2, for comparison.
print("Answer key (exact search, cosine similarity)")
for i, score in exact_top3(question):
    print(f"   score {score:.3f}   {pieces[i].metadata['file']}   year {pieces[i].metadata['year']}")

## Step 5. Score them on ten questions: recall and time

**What this does.** Asks each database ten real support questions and compares its top 3 with the
answer key's top 3. Pieces are matched by their text. It also times each answer.

**What you should see.** Recall@3 of 1.00 for all three, and a middle time of roughly 10 to 50
milliseconds per question each, changing a little from run to run. Almost all of that time is the embedding model turning the question
into numbers, not the database. With 21 pieces, every database compares the question with every
piece, so all three are perfect and the times say nothing about the databases themselves.

In [ ]:
# Ten real questions from FreshCart's support team.
questions = [
    "How many refund claims can a customer make in a rolling thirty day period?",
    "Who bears the cost of a refund when packaging is inadequate for liquid contents?",
    "Within how many hours must photographic evidence of spillage be submitted?",
    "How many working days does it take for an approved refund to be credited?",
    "How long does a partner have to dispute a cost allocation?",
    "What is the status of claim CLM-2024-0314?",
    "What was the refund rate in Q1 2024 according to the QBR?",
    "What is the first response TAT for a P2 case?",
    "What happens to a fourth refund claim within thirty days?",
    "What does spillage mean under the current refund policy?",
]

for name, store in stores.items():
    recalls, times = [], []
    for q in questions:
        # The answer key's top 3, as texts.
        key = {pieces[i].page_content for i, _ in exact_top3(q)}
        # The database's top 3, timed with the stopwatch.
        start = time.perf_counter()
        found = {doc.page_content for doc in store.similarity_search(q, k=3)}
        times.append((time.perf_counter() - start) * 1000)
        # Recall@3: how many of the key's 3 the database also found, out of 3.
        recalls.append(len(key & found) / 3)
    # Average recall, and the middle (median) time in milliseconds.
    print(f"{name:<7} recall@3 {np.mean(recalls):.2f}   median time {np.median(times):.0f} ms per question")

## Step 6. A filter: same idea, three ways to write it

**What this does.** Repeats the Lab 1 fix: search only pieces whose `current` label is `True`, so the
replaced 2022 policy cannot win. Every database can do this, but LangChain does not translate filters,
so each one is written in that database's own language.

- **Chroma:** a dictionary, `{"current": True}`.
- **Qdrant:** a `Filter` object with a condition on `metadata.current`.
- **FAISS:** a dictionary like Chroma's, `{"current": True}`, or a Python function. Behind the scenes
  LangChain fetches 20 nearest pieces and then filters them, so on a large store it can miss results.

**What you should see.** In all three, the 2022 policy is gone and the 2024 policy comes first, then
the next 2024 piece and the SLA table. Same result, three different ways of asking for it.

In [ ]:
# Qdrant's filter objects live in qdrant_client.models.
from qdrant_client import models

# Each database's own way of saying "only current pieces".
filters = {
    "Chroma": {"filter": {"current": True}},
    "Qdrant": {"filter": models.Filter(must=[models.FieldCondition(key="metadata.current",
                                                                    match=models.MatchValue(value=True))])},
    "FAISS": {"filter": {"current": True}},
}

for name, store in stores.items():
    # ** unpacks the dictionary, so each database gets filter=... in its own language.
    results = store.similarity_search_with_score(question, k=3, **filters[name])
    print(name, "with its filter:")
    for doc, score in results:
        print(f"   score {score:.3f}   {doc.metadata['file']}   current {doc.metadata['current']}")
    print()

### Step 6b. Check: filtering after the search quietly returns fewer results

**What this does.** A shortcut people take when a database's filter syntax is awkward: search without a
filter, then throw away the results you may not use. The cell does that for a question about the old
and new policy, asking for 3 results.

**What you should see.** The 3 nearest pieces are the 2022 policy, the 2024 policy and the quarterly
review slides. Dropping the 2022 piece leaves **2 results, not 3**, and nothing warns you. With a filter
that removes more, such as one customer's records out of thousands, the shortcut can return nothing at
all. The database's own filter searches only among the allowed pieces, so it still returns 3.

In [ ]:
# A question the 2022 and 2024 policies both answer, so old pieces crowd the top of the list.
old_and_new = "What does spillage mean, and does it get a full refund?"

# The shortcut: take the 3 nearest pieces, then drop the ones that are not current.
nearest = stores["Qdrant"].similarity_search(old_and_new, k=3)
kept = [doc for doc in nearest if doc.metadata["current"]]
print("asked for 3, filter applied AFTER the search :", len(kept), "result(s)")
for doc in nearest:
    print(f"   {doc.metadata['file']:<32} current {doc.metadata['current']}")

# The database's own filter searches only inside the allowed pieces, so it still returns 3.
inside = stores["Qdrant"].similarity_search(old_and_new, k=3, **filters["Qdrant"])
print("\nasked for 3, the database's own filter       :", len(inside), "result(s)")

## What you found

1. **One method, three databases.** `from_documents` and `similarity_search_with_score` work the same
   on all three, and on 21 pieces they all find exactly what exact search finds.
2. **A score means something different in each database** (Step 4). Read the documentation before
   setting a cut-off on a score, or a cut-off copied from one database will be wrong on another.
3. **21 pieces prove nothing about speed** (Step 5). Every database is fast and perfect at this size.
   The differences appear at hundreds of thousands of pieces, where they stop comparing the question
   with every piece and use a shortcut index instead. The full Lab 2 measures that.
4. **Filters are not portable** (Step 6). The idea is the same, the syntax is not: switching database
   means rewriting every filter.
5. **Filtering after the search loses results, silently** (Step 6b). Always filter inside the
   database.

**Where this goes next.** The full Lab 2 notebook, `lab02_vector_stores`, loads 100,000 vectors, turns
the speed-versus-recall dial, measures compression and the filter at scale, and ends with a memo
recommending one database.

## Look it up yourself: every setting the libraries offer

**What this does.** For the main things this lab calls, the cell prints every parameter (read from the
installed library), the values this lab passed, and one setting worth trying.

**What you should see.** Five short blocks. Notice that each database takes different connection and
comparison settings: that is where they differ most.

In [ ]:
# inspect reads the list of parameters a function or class accepts; textwrap folds long lines.
import inspect, textwrap

# parameters(thing): the parameters `thing` accepts, as text such as "k=4".
def parameters(thing):
    found = []
    # Some classes list their settings as "fields", not on __init__.
    if hasattr(thing, "model_fields"):
        return [n if f.is_required() else f"{n}={f.default!r}" for n, f in thing.model_fields.items()]
    # A class keeps its parameters on __init__, and maybe on its parents' __init__ as well.
    places = [c.__init__ for c in thing.__mro__ if "__init__" in vars(c)] if inspect.isclass(thing) else [thing]
    for place in places:
        try:
            sig = inspect.signature(place)
        except (TypeError, ValueError):       # some built-in objects cannot be read; skip them
            continue
        for p in sig.parameters.values():
            # Skip "self", and the *args / **kwargs catch-alls, which are not named settings.
            if p.name in ("self", "cls") or p.kind in (p.VAR_POSITIONAL, p.VAR_KEYWORD):
                continue
            text = p.name if p.default is p.empty else f"{p.name}={p.default!r}"
            if p.name not in [f.split("=")[0] for f in found]:
                found.append(text)
        # Stop climbing to parent classes once a level has no **kwargs to pass settings on.
        if not any(p.kind == p.VAR_KEYWORD for p in sig.parameters.values()):
            break
    return found

# look_up prints the three lines for one function or class.
def look_up(name, thing, this_lab_passed, worth_a_look):
    print(name)
    print(textwrap.fill("every parameter : " + ", ".join(parameters(thing)), 110, subsequent_indent=" " * 18))
    print("this lab passed :", this_lab_passed)
    print("worth a look    :", worth_a_look, "\n")

look_up("Chroma", Chroma, 'collection_name="lab02", collection_metadata={"hnsw:space": "cosine"}',
        "persist_directory: a folder, so the pieces are still there after a restart")
look_up("QdrantVectorStore.from_texts", QdrantVectorStore.from_texts, 'location=":memory:", collection_name="lab02" (from_documents passes them on here)',
        "url: the address of a real Qdrant server instead of memory")
look_up("FAISS", FAISS, "nothing: the defaults",
        "distance_strategy: how it compares (the default is distance, lower means closer)")
look_up("store.similarity_search_with_score (Chroma)", Chroma.similarity_search_with_score, "query, k=3, filter=...",
        "filter: Chroma's own filter language; Qdrant has its own, FAISS another")
look_up("FastEmbedEmbeddings", FastEmbedEmbeddings, 'model_name = "BAAI/bge-small-en-v1.5", the default',
        "batch_size: how many pieces it turns into numbers at once")

## Your notes

Double click to type: which database you would try first for FreshCart, and why.